# Functional tests

Below is a list of functional tests which should be included in sending and receiving implementations.

### Receiving

- Ensure the public key can be extracted from non-standard P2PKH scriptSigs
- Ensure taproot script path spends are included, using the taproot output key (unless H is used as the taproot internal key)
- Ensure the scanner can extract the public key from each of the input types supported (e.g. P2WPKH, P2SH-P2WPKH, etc.)

In [13]:
import importlib
import sys

# rimuovi i moduli dalla cache per forzare il reload
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['receive', 'utils']):
        del sys.modules[mod]

# ora reimporta
from receive import receiving_run

In [14]:
import json
import sys
import os
import traceback
import copy
from typing import Tuple, List, Dict, Any

import importlib
import receive
importlib.reload(receive)
from receive import receiving_run

# --- path setup robusto ---
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Jupyter fallback
    BASE_DIR = os.getcwd()

SRC_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
sys.path.insert(0, SRC_DIR)

TEST_VECTORS_PATH = os.path.join(BASE_DIR, 'test_vectors.json')

from receive import receiving_run


# =========================================================
# Utils
# =========================================================

def deep_clone(x):
    return copy.deepcopy(x)


def normalize_outputs(outputs: List[Dict]) -> List[tuple]:
    """Normalizza per confronto deterministico"""
    return sorted(
        (o['pub_key'], o['priv_key_tweak'])
        for o in outputs
    )


def validate_case_structure(test_id: int, case_idx: int, receiving: dict):
    """Fail early se il JSON è malformato"""
    if 'given' not in receiving or 'expected' not in receiving:
        raise ValueError(f"[{test_id}:{case_idx}] missing given/expected")

    given = receiving['given']
    expected = receiving['expected']

    required_given = ['vin', 'outputs', 'key_material']
    for k in required_given:
        if k not in given:
            raise ValueError(f"[{test_id}:{case_idx}] missing given.{k}")

    if 'addresses' not in expected or 'outputs' not in expected:
        raise ValueError(f"[{test_id}:{case_idx}] missing expected fields")


# =========================================================
# Core test logic
# =========================================================

def run_receiving_test(test_id: int, raw_data: dict) -> Tuple[bool, str]:
    data = deep_clone(raw_data)  # 🔒 isolamento totale

    comment = data.get('comment', f'test #{test_id}')
    receiving_cases = data.get('receiving', [])

    if not receiving_cases:
        return True, 'SKIP (nessun caso receiving)'

    all_passed = True
    details = []

    for case_idx, receiving in enumerate(receiving_cases):

        try:
            validate_case_structure(test_id, case_idx, receiving)
        except Exception as e:
            return False, f"Struttura invalida: {e}"

        given = receiving['given']
        expected = receiving['expected']

        vin = deep_clone(given.get('vin'))
        outputs = deep_clone(given.get('outputs', []))
        key_material = deep_clone(given.get('key_material'))
        labels = deep_clone(given.get('labels'))

        expected_addresses = expected.get('addresses', [])
        expected_outputs = expected.get('outputs', [])

        # 🔍 snapshot pre-esecuzione (per rilevare mutazioni)
        snapshot = {
            "vin": deep_clone(vin),
            "outputs": deep_clone(outputs),
            "key_material": deep_clone(key_material),
            "labels": deep_clone(labels),
        }

        # -------------------------------------------------
        # ESECUZIONE
        # -------------------------------------------------
        try:
            addresses, wallet = receiving_run(
                vin=vin,
                outputs=outputs,
                key_material=key_material,
                labels=labels if labels else None,
            )
        except Exception as e:
            all_passed = False
            details.append(f'  case {case_idx}: EXCEPTION — {e}')
            details.append(traceback.format_exc())
            continue

        # -------------------------------------------------
        # 🔒 CHECK MUTAZIONI
        # -------------------------------------------------
        mutated = []
        if vin != snapshot["vin"]:
            mutated.append("vin")
        if outputs != snapshot["outputs"]:
            mutated.append("outputs")
        if key_material != snapshot["key_material"]:
            mutated.append("key_material")
        if labels != snapshot["labels"]:
            mutated.append("labels")

        if mutated:
            details.append(f'  case {case_idx}: ⚠️ MUTATION DETECTED → {mutated}')

        # -------------------------------------------------
        # ✔ CONFRONTI
        # -------------------------------------------------

        addr_ok = sorted(addresses) == sorted(expected_addresses)

        out_ok = (
            normalize_outputs(wallet) ==
            normalize_outputs(expected_outputs)
        )

        passed = addr_ok and out_ok

        if not passed:
            all_passed = False

            if not addr_ok:
                details.append(f'  case {case_idx}: ADDR MISMATCH')
                details.append(f'    got:      {addresses}')
                details.append(f'    expected: {expected_addresses}')

            if not out_ok:
                details.append(f'  case {case_idx}: OUTPUTS MISMATCH')

                details.append('    got wallet:')
                for w in wallet:
                    details.append(
                        f'      pub={w.get("pub_key")} '
                        f'tweak={w.get("priv_key_tweak")}'
                    )

                details.append('    expected:')
                for e in expected_outputs:
                    details.append(
                        f'      pub={e.get("pub_key")} '
                        f'tweak={e.get("priv_key_tweak")}'
                    )
        else:
            details.append(f'  case {case_idx}: OK')

    return all_passed, '\n'.join(details)


# =========================================================
# Runner
# =========================================================

def load_vectors(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Test vectors not found: {path}")
    with open(path, 'r') as f:
        return json.load(f)


def main():
    
    import importlib
    import receive as receive_module
    importlib.reload(receive_module)
    from receive import receiving_run as _receiving_run
    
    vectors = load_vectors(TEST_VECTORS_PATH)

    total = len(vectors)
    passed = 0
    failed = []

    col = max((len(v.get('comment', '')) for v in vectors), default=10) + 2

    print(f"\n{'='*70}")
    print(f"  Silent Payments — Receiving test suite ({total} vettori)")
    print(f"{'='*70}\n")

    for i, raw_data in enumerate(vectors):
        ok, detail = run_receiving_test(i, raw_data)

        status = '✓ PASS' if ok else '✗ FAIL'
        comment = raw_data.get('comment', f'test #{i}')

        print(f"  [{i:>2}] {comment:<{col}} {status}")

        if not ok:
            print(detail)
            failed.append(i)
        else:
            passed += 1

    print(f"\n{'='*70}")
    print(f"  Risultato: {passed}/{total} passati", end='')
    if failed:
        print(f"  —  falliti: {failed}")
    else:
        print("  🎉")
    print(f"{'='*70}\n")

    sys.exit(0 if not failed else 1)


if __name__ == '__main__':
    main()



  Silent Payments — Receiving test suite (26 vettori)

DEBUG VIN:  [{'txid': 'f4184fc596403b9d638783cf57adfe4c75c605f6356fbc91338530e9831e9e16', 'vout': 0, 'scriptSig': '483046022100ad79e6801dd9a8727f342f31c71c4912866f59dc6e7981878e92c5844a0ce929022100fb0d2393e813968648b9753b7e9871d90ab3d815ebf91820d704b19f4ed224d621025a1e61f898173040e20616d43e9f496fba90338a39faa1ed98fcbaeee4dd9be5', 'txinwitness': '', 'prevout': {'scriptPubKey': {'hex': '76a91419c2f3ae0ca3b642bd3e49598b8da89f50c1416188ac'}}}, {'txid': 'a1075db55d416d3ca199f55b6084e2115b9345e16c5cf302fc80e9d5fbf5d48d', 'vout': 0, 'scriptSig': '48304602210086783ded73e961037e77d49d9deee4edc2b23136e9728d56e4491c80015c3a63022100fda4c0f21ea18de29edbce57f7134d613e044ee150a89e2e64700de2d4e83d4e2103bd85685d03d111699b15d046319febe77f8de5286e9e512703cdee1bf3be3792', 'txinwitness': '', 'prevout': {'scriptPubKey': {'hex': '76a914d9317c66f54ff0a152ec50b1d19c25be50c8e15988ac'}}}]
DEBUG TX:  {'txid': 'f4184fc596403b9d638783cf57adfe4c75c605f6356fbc91

SystemExit: 0